# Dataset Creation for Linear AE
This notebook reproduces the same data preparation logic used in Step 1 and Step 2 of `05_linearAE.ipynb`:
- load correlation matrices from a `.pt` file
- select only matrices in the range `[MATRIX_START, MATRIX_END)`
- split matrices into train, validation, and test sets
- save the three datasets under `data/processed/dataset`

In [10]:
from pathlib import Path
import sys
import json

import numpy as np
import torch

np.set_printoptions(suppress=True, precision=4)

## Step 1: Load Data and Select Matrix Range
Load correlation matrices from a `.pt` file and keep only matrices in `[MATRIX_START, MATRIX_END)` (same behavior as `05_linearAE.ipynb`).

In [ ]:
MY_FILE_NAME = 'data_00_20_w724_s10.pt'

if 'google.colab' in sys.modules:
    print('Environment detected: Google Colab')
    IS_COLAB = True
else:
    print('Environment detected: Local (PC)')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    processed_dir = Path('/content/drive/MyDrive/dataset_tesi')
else:
    project_root = Path.cwd().resolve().parent
    processed_dir = project_root / 'data' / 'processed' / 'correlation_matrices'

selected_file = processed_dir / MY_FILE_NAME
if not selected_file.exists():
    raise FileNotFoundError(f"File '{MY_FILE_NAME}' not found in: {processed_dir.resolve()}")

# Select the matrix range to use: [MATRIX_START, MATRIX_END)
MATRIX_START = 49  # number or None
MATRIX_END = None  # number or None; e.g., 500 to use only first 500 matrices

print(f'Selected file: {selected_file.name}')
print(f'Matrix range requested: [{MATRIX_START}, {MATRIX_END})')

Environment detected: Local (PC)
Selected file: corr_windows_tensor_data_00_20_w724_s10.pt
Matrix range requested: [49, None)


In [12]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')

    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')

    return corr_tensor.float(), meta


corr_tensor, meta = load_corr_payload(selected_file)

orig_n = corr_tensor.shape[0]
start_idx = 0 if MATRIX_START is None else int(MATRIX_START)
end_idx = orig_n if MATRIX_END is None else int(MATRIX_END)

if not (0 <= start_idx < end_idx <= orig_n):
    raise ValueError(f'Invalid matrix range [{start_idx}, {end_idx}) for dataset size {orig_n}')

corr_tensor = corr_tensor[start_idx:end_idx]
if 'window_ranges' in meta and len(meta['window_ranges']) == orig_n:
    meta['window_ranges'] = meta['window_ranges'][start_idx:end_idx]

print(f'corr_tensor shape (selected range): {tuple(corr_tensor.shape)}')
print(f'using matrices: [{start_idx}, {end_idx}) out of {orig_n}')
print(f'dtype: {corr_tensor.dtype}')

corr_tensor shape (selected range): (412, 362, 362)
using matrices: [49, 461) out of 461
dtype: torch.float32


## Step 2: Create Train/Validation/Test Splits
Keep each sample as a full `N x N` correlation matrix, shuffle with a fixed seed, and split into train/val/test (same logic used in `05_linearAE.ipynb`).

In [13]:
SEED = 42
VAL_FRACTION = 0.2
TEST_FRACTION = 0.1

if not (0.0 < VAL_FRACTION < 1.0):
    raise ValueError('VAL_FRACTION must be in (0, 1).')
if not (0.0 < TEST_FRACTION < 1.0):
    raise ValueError('TEST_FRACTION must be in (0, 1).')
if VAL_FRACTION + TEST_FRACTION >= 1.0:
    raise ValueError('VAL_FRACTION + TEST_FRACTION must be < 1.0.')

corr_np = corr_tensor.numpy().astype(np.float32)
n_matrices, n_assets, _ = corr_np.shape

rng = np.random.default_rng(SEED)
indices = np.arange(n_matrices)
rng.shuffle(indices)

n_val = max(1, int(n_matrices * VAL_FRACTION))
n_test = max(1, int(n_matrices * TEST_FRACTION))
n_train = n_matrices - n_val - n_test

if n_train <= 0:
    raise ValueError('Training split is empty. Reduce VAL_FRACTION/TEST_FRACTION or use more matrices.')

val_idx = indices[:n_val]
test_idx = indices[n_val:n_val + n_test]
train_idx = indices[n_val + n_test:]

train_corr = torch.from_numpy(corr_np[train_idx])
val_corr = torch.from_numpy(corr_np[val_idx])
test_corr = torch.from_numpy(corr_np[test_idx])

print(f'Number of matrices: {n_matrices}')
print(f'Matrix shape: ({n_assets}, {n_assets})')
print(f'Train tensor shape: {tuple(train_corr.shape)}')
print(f'Val tensor shape: {tuple(val_corr.shape)}')
print(f'Test tensor shape: {tuple(test_corr.shape)}')
print(f'Train size: {len(train_idx)} | Val size: {len(val_idx)} | Test size: {len(test_idx)}')

Number of matrices: 412
Matrix shape: (362, 362)
Train tensor shape: (289, 362, 362)
Val tensor shape: (82, 362, 362)
Test tensor shape: (41, 362, 362)
Train size: 289 | Val size: 82 | Test size: 41


## Step 3: Save Train/Validation/Test Datasets
Save the three `.pt` files and the summary `.json` inside a dedicated subfolder under `data/processed/dataset`.

In [14]:
dataset_root_dir = processed_dir / 'dataset'
dataset_root_dir.mkdir(parents=True, exist_ok=True)

range_tag = f'range_{start_idx}_{end_idx}'
split_tag = f'seed{SEED}_val{int(VAL_FRACTION * 100)}_test{int(TEST_FRACTION * 100)}'
base_name = selected_file.stem
subfolder_name = f'{base_name}_{range_tag}_{split_tag}'
dataset_dir = dataset_root_dir / subfolder_name
dataset_dir.mkdir(parents=True, exist_ok=True)

split_payloads = {
    'train': {
        'corr_tensor': train_corr.clone(),
        'indices': train_idx.tolist(),
    },
    'val': {
        'corr_tensor': val_corr.clone(),
        'indices': val_idx.tolist(),
    },
    'test': {
        'corr_tensor': test_corr.clone(),
        'indices': test_idx.tolist(),
    },
}

common_meta = {
    'source_file': str(selected_file),
    'matrix_range': {'start_idx': int(start_idx), 'end_idx': int(end_idx)},
    'matrix_shape': [int(n_assets), int(n_assets)],
    'split_fractions': {'val_fraction': float(VAL_FRACTION), 'test_fraction': float(TEST_FRACTION)},
    'seed': int(SEED),
    'subfolder_name': subfolder_name,
}

saved_paths = {}
for split_name, payload in split_payloads.items():
    output_path = dataset_dir / f'{split_name}.pt'
    torch.save(
        {
            **payload,
            'split': split_name,
            'meta': {
                **common_meta,
                'split_size': int(payload['corr_tensor'].shape[0]),
            },
        },
        output_path,
    )
    saved_paths[split_name] = output_path

summary_path = dataset_dir / 'summary.json'
summary_payload = {
    'subfolder_name': subfolder_name,
    'dataset_dir': str(dataset_dir),
    'files': {k: str(v) for k, v in saved_paths.items()},
    'sizes': {
        'train': int(len(train_idx)),
        'val': int(len(val_idx)),
        'test': int(len(test_idx)),
    },
    'meta': common_meta,
}

with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_payload, f, indent=4)

print('Saved dataset files inside subfolder:')
for split_name, output_path in saved_paths.items():
    print(f' - {split_name}: {output_path}')
print(f' - summary: {summary_path}')

Saved dataset files inside subfolder:
 - train: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\dataset\corr_windows_tensor_data_00_20_w724_s10_range_49_461_seed42_val20_test10\train.pt
 - val: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\dataset\corr_windows_tensor_data_00_20_w724_s10_range_49_461_seed42_val20_test10\val.pt
 - test: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\dataset\corr_windows_tensor_data_00_20_w724_s10_range_49_461_seed42_val20_test10\test.pt
 - summary: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\dataset\corr_windows_tensor_data_00_20_w724_s10_range_49_461_seed42_val20_test10\summary.json
